In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import *
from pyspark.sql.types import *

**Question : Daily Completed Revenue by Category (with Data Quality Issues)**

Scenario: You're given a raw orders export (orders.csv) from an e-commerce system. As is typical in real pipelines, it has some dirty data — missing quantity, missing order_date, and non-COMPLETED orders that shouldn't count toward revenue.

**Problem:**

- Read the CSV with a proper schema (don't just let Spark infer it — define it explicitly).
- Drop rows where quantity or order_date is null (bad records — in a real pipeline you'd route these to a quarantine/rejects table, but for this exercise just drop them).
- Keep only rows where status = 'COMPLETED'.
- Compute revenue = quantity * unit_price.
- Return total revenue per product_category, sorted by revenue descending.

**Schema (orders.csv)**

| Column | Type |
| --- | --- |
| order_id | int |
| customer_id | string |
| order_date | date |
| product_category | string |
| quantity | int |
| unit_price | double |
| status | string |

**Expected Output**

| product_category | total_revenue |
| --- | --- |
| Electronics | 1799.00 |
| Furniture | 1050.00 |
| Clothing | 150.00 |
| Grocery | 86.50 |

In [0]:
schema = StructType(
    [
        StructField("order_id", IntegerType()),
        StructField("customer_id", StringType()),
        StructField("order_date", DateType()),
        StructField("product_category", StringType()),
        StructField("quantity", IntegerType()),
        StructField("unit_price", DecimalType(20, 5)),
        StructField("status", StringType()),
    ]
)


In [0]:
orders_df = (
    spark.read.format("csv")
    .option("header", True)
    .schema(schema)
    .load("/Workspace/Users/jeevan.azureacc2@gmail.com/spark-practice/data/orders.csv")
)

In [0]:
orders_df = orders_df.filter(
    (col("quantity").isNotNull()) 
    & (col("order_date").isNotNull())
    & (col("status") == "COMPLETED")
).withColumn(
    "revenue",
    col("quantity") * col("unit_price")
)

In [0]:
output_df = (
    orders_df.groupBy(col("product_category"))
    .agg(sum(col("revenue")).alias("total_revenue"))
)
output_df.orderBy(col("total_revenue").desc()).show()